# Binary morphology

Compare erosion and dilation using cross and square structuring elements, then measure connected shapes.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/image_processing").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from image_processing import load_sample, show_images

try:
    import cv2 as cv
except ImportError:
    cv = None
    print("OpenCV is unavailable. NumPy examples still run; install requirements.txt for comparisons.")

## Binary input and structuring element

Foreground is True. Pixels outside the image are treated as background in both implementations.

In [ ]:
from image_processing import binary_morphology
image = load_sample("binary")
binary = image > 128
cross = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)
show_images([binary, cross], ["Binary sample", "Cross element"])

## Erosion and dilation from scratch

Erosion keeps a pixel when all selected neighbors are foreground. Dilation keeps it when any selected neighbor is foreground.

In [ ]:
eroded = binary_morphology(binary, cross, "erode")
dilated = binary_morphology(binary, cross, "dilate")
show_images([binary, eroded, dilated], ["Original", "Erosion", "Dilation"])
if cv is not None:
    kernel = cross.astype(np.uint8)
    np.testing.assert_array_equal(eroded, cv.erode(binary.astype(np.uint8), kernel, borderType=cv.BORDER_CONSTANT, borderValue=0).astype(bool))
    np.testing.assert_array_equal(dilated, cv.dilate(binary.astype(np.uint8), kernel, borderType=cv.BORDER_CONSTANT, borderValue=0).astype(bool))

## Larger elements

A larger element changes the scale of erosion and dilation. Each operation starts from the same original input.

In [ ]:
square = np.ones((11, 11), dtype=bool)
show_images([binary_morphology(binary, square, "erode"), binary_morphology(binary, square, "dilate")], ["11x11 erosion", "11x11 dilation"])

## Area and centroid

Connected-component area counts foreground pixels. Counting contour vertices does not measure object area.

In [ ]:
if cv is not None:
    count, labels, stats, centroids = cv.connectedComponentsWithStats(binary.astype(np.uint8))
    for label in range(1, count):
        print("Object", label, "pixel area:", stats[label, cv.CC_STAT_AREA], "centroid:", centroids[label])